# make_core_metadata_template: Write an empty core-metadata sheet for a TMA grid (e.g. 10 rows x 20 columns = 200 cores).

```text
python scripts/make_core_metadata_template.py --slide-id TMA2 --rows 10 --cols 20         --extra grade stage msi_status --out config/core_metadata_TMA2.csv

Open the CSV in Excel, fill one row per core (patient_id, organ, tissue_type, ...), set
include=FALSE for cores to leave out, and keep it as CSV. Several slides: run once per
slide and concatenate, or pass --append to add to an existing sheet.
Then check it:  python scripts/make_core_metadata_template.py --check config/core_metadata_TMA2.csv
```

**Notebook version of `tma_toolkit/scripts/make_core_metadata_template.py`.** Same code, laid out for interactive work:

1. Set the options in the **Parameters** cell (the script's command-line options: `--slide-id` becomes `slide_id`).
2. Run the cells in order. After each cell you can inspect its variables, tables and images.

Use the `.py` script for batch jobs on the cluster; both produce the same outputs. This notebook is generated by `tools/make_notebooks.py`, which overwrites it, so save your own edits under another name.

In [ ]:
import os

# folder of make_core_metadata_template.py; this notebook sits in notebooks/ next to it
SCRIPT_DIR = os.path.abspath('../scripts')

# Which Python runs this notebook? It must be the conda environment, not the system one.
import sys
print('Python:', sys.executable)
try:
    import numpy  # noqa: F401
except ModuleNotFoundError:
    raise ModuleNotFoundError(
        'This kernel is not the exp1-imaging environment. In a terminal, once:\n'
        '  conda activate exp1-imaging\n'
        '  python -m ipykernel install --user --name exp1-imaging --display-name exp1-imaging\n'
        'then in Jupyter: Kernel > Change kernel > exp1-imaging  (environment file: experiment1_tma/env/py_imaging.yml + pip install -e tma_toolkit)')

In [ ]:
import argparse
import os
import sys

import pandas as pd

sys.path.insert(0, os.path.dirname(SCRIPT_DIR))
from tma_toolkit.metadata import make_template, read_core_metadata  # noqa: E402

## Parameters

In [ ]:
from types import SimpleNamespace

# Same options as `python tma_toolkit/scripts/make_core_metadata_template.py --help` (--slide-id -> slide_id).

# Options marked REQUIRED have no default: replace their None, e.g. slide_id="TMA_ORGAN".
# Paths are plain strings in quotes.

a = SimpleNamespace(
    slide_id=None,
    tma_id=None,    # prefix of core ids (default = slide id)
    rows=None,
    cols=None,
    extra=[],       # additional metadata columns. a list, e.g. ['A01', 'B03']
    out=None,
    append=False,
    check=None,     # validate an existing sheet and print a summary
)

## Run

In [ ]:
if a.check:
    df = read_core_metadata(a.check)
    print(f"{len(df)} cores, {df.slide_id.nunique()} slide(s)")
    for c in ("organ", "tissue_type", "diagnosis", "include"):
        if c in df:
            print(df[c].value_counts(dropna=False).to_string(), "\n")
    if "patient_id" in df:
        print(f"{df.patient_id.nunique()} patients; cores per patient: "
              f"{df.groupby('patient_id').size().describe()[['min', 'mean', 'max']].round(1).to_dict()}")
else:
    t = make_template(a.slide_id, a.rows, a.cols, a.tma_id, a.extra)
    if a.append and os.path.exists(a.out):
        t = pd.concat([pd.read_csv(a.out), t], ignore_index=True)
    t.to_csv(a.out, index=False)
    print(f"wrote {len(t)} rows -> {a.out}")